# Document to PDF

**What it does**
Choose a document, click **Convert to PDF**, and get the PDF: the app shows where it is saved and a link to download it.

| Format | Status |
|---|---|
| Word (.docx) | works |
| Word (old .doc), PDF, Excel, Outlook (.msg) | can be chosen, not converted yet |

**Why**
Gemini and Claude read PDF, not Word. The machine in ING has Jupyter but no web page can be opened, so this runs inside the notebook: no web server, no open port.

**How to use**
1. Run all cells (Run All). The app appears below the last cell.
2. **Choose files**, then **Convert to PDF**. A spinner shows while it works.
3. Copy the path shown, or click **Download**. The PDFs are saved under `storage/converted/<date-time>/`.

**Engine**: LibreOffice when it is installed (exact Word layout); otherwise python-docx + reportlab (content kept; columns, text boxes, headers and footers dropped).

**Needs**: `ipywidgets`, `python-docx`, `reportlab` (see `requirements.txt`).

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.config import load_env_file, settings
load_env_file()
settings.cache_clear()
from services.evidences import libreoffice
from services.presentation.converter_app import ConverterApp
print("PDFs go to:", settings().storage_root / "converted")
print("LibreOffice:", libreoffice.binary() or "not installed, the python engine is used")

## Check: is LibreOffice on this machine?

Run this first. It looks for LibreOffice (on `PATH` and in the usual install folders on Linux, macOS and Windows), asks its version, and converts a small fake Word file with it to prove it really works.

- **OK**: Word keeps its exact layout; Excel and Outlook to PDF are possible later.
- **Not found / failed**: the app still converts Word with the python engine (content kept, exact layout not). To install on Linux: `apt install libreoffice-writer libreoffice-calc` (ask IT).

In [ ]:
import platform, subprocess, tempfile
from pathlib import Path
from services.evidences import libreoffice
from services.evidences.docx_converter import DocxConverter

print("Machine:", platform.system(), platform.release())
program = libreoffice.binary()
if not program:
    print("LibreOffice: NOT FOUND")
    print("Looked on PATH (soffice, libreoffice) and in:")
    for place in libreoffice.KNOWN_PLACES:
        print("   ", place)
    print("=> Word is converted with the python engine.")
else:
    print("LibreOffice: found at", program)
    try:
        version = subprocess.run([program, "--version"], capture_output=True, text=True, timeout=60)
        print("Version:", (version.stdout or version.stderr).strip() or "(no answer)")
    except Exception as exc:
        print("Version: could not ask:", exc)
    try:
        import docx
        folder = Path(tempfile.mkdtemp())
        sample = folder / "libreoffice_check.docx"
        document = docx.Document()
        document.add_paragraph("LibreOffice check")
        document.save(sample)
        pdf = DocxConverter(engine="libreoffice").convert(sample).pdf
        print(f"Test conversion: OK ({pdf.stat().st_size} bytes) -> {pdf}")
        print("=> Word is converted with LibreOffice (exact layout).")
    except Exception as exc:
        print("Test conversion: FAILED:", exc)
        print("=> LibreOffice is there but does not work here; the app would still try it and fail.")
        print("   Use ConverterApp(engine='python') in the last cell.")

In [ ]:
ConverterApp()